This notebook will anonimize the generated data in the simulated data folder. It will get all the images in that folder, give them a random subject ID and resave them in a new folder called "anonimized_simulated_data". It will also generate a csv file with the mapping of the original file names to the new anonimized file names. This will allow us to keep track of the data while also ensuring that the data is anonimized for any future use.

In [20]:
# import necessary libraries
import os
import random
import shutil

# define the path to the simulated data folder and the new folder for anonimized data
simulated_data_folder_full_reports = "simulated_data/full_reports"
simulated_data_folder_erp = "simulated_data/erp_image"
simulated_data_folder_itpc = "simulated_data/itpc_image"
anonimized_data_folder = "anonimized_simulated_data"

# create the new folder if it doesn't exist
if not os.path.exists(anonimized_data_folder):
    os.makedirs(anonimized_data_folder)


In [37]:
# get all the files in the simulated data folder
files = os.listdir(simulated_data_folder_full_reports)
files_erp = os.listdir(simulated_data_folder_erp)
files_itpc = os.listdir(simulated_data_folder_itpc)
# we are going to do this for ERP and full report images separately because the file names have different formats and we want to make sure we are correctly identifying the conditions and subject numbers for each type of image. The new subject IDs will be the same for both types of images to maintain the mapping between the ERP and full report images for each subject.

# create a list to store the mapping of original file names to new anonimized file names
mapping = []

# define the conditions that need to be identified in the file names and the corresponding new file name format
conditions = {
    "reduced": "reduced_sub",
    "reduced delayed": "reduced_and_delayed",
    "delayed": "delayed_sub",
    "normal": "normal_sub",
    "abolished": "abolished_sub",
}


# generate a set of random IDs that have already been used to ensure uniqueness
ids = list(range(1000, 10000))
random.shuffle(ids)


# loop through the files and rename them with a random subject ID
for file in files:
    if file.endswith(".jpg") and "full_report" in file:
        # generate a random subject ID from the list of IDs and remove it from the list to ensure uniqueness
        subject_id = ids.pop()

        # get condition from the original file name
        for condition in conditions:
            # we have a problem with reduced_delayed and delayed
            if condition in file: # if condition is found in file name
                condition_name = conditions[condition]
                # if the condition is found to be reduced, check if its reduced_delayed or just reduced.
                if condition == "reduced":
                    if "delayed" in file:
                        condition_name = conditions["reduced delayed"]
                    else:
                        condition_name = conditions["reduced"]
                break

        else:
            condition_name = "unknown_condition"

        # get subject number from the original file name
        subject_number = file.split("_")[-1]
        # get rid of the leading "sub" and the trailing ".jpg"
        subject_number = subject_number.replace("sub", "").replace(".jpg", "")



        # create the new file name
        new_file_name_full_report = f"subject_{subject_id}_full_report.jpg"
        new_file_name_erp = f"subject_{subject_id}_erp.jpg"
        new_file_name_itpc = f"subject_{subject_id}_itpc.jpg"


        # copy full report to anonimized full report folder
        shutil.copy(os.path.join(simulated_data_folder_full_reports, file), os.path.join(anonimized_data_folder,'full_report',new_file_name_full_report))

        # finding corresponding ERP image
        for file_erp in files_erp:
            if file_erp.endswith(".jpg") and "ERP" in file_erp:
                # get condition from the original file name
                for condition in conditions:
                    if condition in file_erp:
                        condition_name_erp = conditions[condition]
                        # if the condition is found to be reduced, check if its reduced_delayed or just reduced.
                        if condition == "reduced":
                            if "delayed" in file_erp:
                                condition_name_erp = conditions["reduced delayed"]
                            else:
                                condition_name_erp = conditions["reduced"]
                        break
                else:
                    condition_name_erp = "unknown_condition"

                # get subject number from the original file name
                subject_number_erp = file_erp.split("_")[-1]
                # get rid of the leading "sub" and the trailing ".jpg"
                subject_number_erp = subject_number_erp.replace("sub", "").replace(".jpg", "")

                # check if the condition and subject number match with the current full report image
                if condition_name_erp == condition_name and subject_number_erp == subject_number:
                    # copy the ERP image to the new folder with the new name
                    shutil.copy(os.path.join(simulated_data_folder_erp, file_erp), os.path.join(anonimized_data_folder,'erp_image',new_file_name_erp))
                    break

        # finding corresponding ITPC image
        for file_itpc in files_itpc:
            if file_itpc.endswith(".jpg") and "ITPC" in file_itpc:
                # get condition from the original file name
                for condition in conditions:
                    if condition in file_itpc:
                        condition_name_itpc = conditions[condition]
                        # if the condition is found to be reduced, check if its reduced_delayed or just reduced.
                        if condition == "reduced":
                            if "delayed" in file_itpc:
                                condition_name_itpc = conditions["reduced delayed"]
                            else:
                                condition_name_itpc = conditions["reduced"]
                        break
                else:
                    condition_name_itpc = "unknown_condition"

                # get subject number from the original file name
                subject_number_itpc = file_itpc.split("_")[-1]
                # get rid of the leading "sub" and the trailing ".jpg"
                subject_number_itpc = subject_number_itpc.replace("sub", "").replace(".jpg", "")

                # check if the condition and subject number match with the current full report image
                if condition_name_itpc == condition_name and subject_number_itpc == subject_number:
                    # copy the ITPC image to the new folder with the new name
                    shutil.copy(os.path.join(simulated_data_folder_itpc, file_itpc), os.path.join(anonimized_data_folder,'itpc_image',new_file_name_itpc))
                    break





        # store mapping of original file name to new anonimized file name in the mapping list with the condition and original subject number for both the full report, ERP images and ITPC images.
        mapping.append({"condition": condition_name, "original_subject_number": subject_number, "original_file_name_full_report": file, "anonimized_file_name_full_report": new_file_name_full_report, "original_file_name_erp": file_erp, "anonimized_file_name_erp": new_file_name_erp, "original_file_name_itpc": file_itpc, "anonimized_file_name_itpc": new_file_name_itpc})




In [38]:
# save the mapping to a csv file
import pandas as pd
mapping_df = pd.DataFrame(mapping)
mapping_df.to_csv(os.path.join(anonimized_data_folder, "mapping.csv"), index=False)
